# 🚗 Car Price Prediction Using Machine Learning

**Objective:** Predict the selling price of a used car using car age, brand, present price, kilometers driven, fuel type, seller type, transmission, and owner information.

This notebook uses two regression models: **Linear Regression** and **Random Forest Regressor**.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load Dataset

Download `car data.csv` from the Vehicle Dataset from CarDekho and keep it in the same folder as this notebook.

In [ ]:
df = pd.read_csv('car data.csv')
df.head()

## 3. Understand the Dataset

In [1]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

print('\nData types and non-null values:')
df.info()

print('\nSummary statistics:')
display(df.describe(include='all').T)

NameError: name 'df' is not defined

## 4. Data Cleaning

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip()

# Remove duplicate rows
print('Duplicate rows before:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Duplicate rows after:', df.duplicated().sum())

# Check missing values
print('\nMissing values:')
print(df.isnull().sum())

# Clean categorical values
for col in ['Fuel_Type', 'Seller_Type', 'Transmission']:
    df[col] = df[col].astype(str).str.strip().str.title()

print('\nUnique categorical values:')
for col in ['Fuel_Type', 'Seller_Type', 'Transmission']:
    print(col, df[col].unique())

## 5. Feature Engineering

Create **Car_Age** from `Year` and extract **Brand** from `Car_Name`.

In [ ]:
CURRENT_YEAR = 2026

df['Car_Age'] = CURRENT_YEAR - df['Year']
df['Brand'] = df['Car_Name'].astype(str).str.strip().str.split().str[0].str.title()

display(df[['Car_Name', 'Brand', 'Year', 'Car_Age']].head())

## 6. Exploratory Data Analysis

In [ ]:
# Distribution of selling prices
plt.figure(figsize=(8, 5))
sns.histplot(df['Selling_Price'], kde=True)
plt.title('Distribution of Selling Price')
plt.xlabel('Selling Price')
plt.ylabel('Number of Cars')
plt.show()

In [ ]:
# Selling price vs fuel type
plt.figure(figsize=(8, 5))
sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df)
plt.title('Selling Price vs Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Selling Price')
plt.show()

In [ ]:
# Selling price vs car age
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Car_Age', y='Selling_Price', data=df)
plt.title('Selling Price vs Car Age')
plt.xlabel('Car Age (years)')
plt.ylabel('Selling Price')
plt.show()

## 7. Feature Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

## 8. Select Features and Target

In [ ]:
features = [
    'Brand', 'Car_Age', 'Present_Price', 'Kms_Driven',
    'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner'
]

X = df[features]
y = df['Selling_Price']

categorical_features = ['Brand', 'Fuel_Type', 'Seller_Type', 'Transmission']
numerical_features = ['Car_Age', 'Present_Price', 'Kms_Driven', 'Owner']

## 9. Encode Categorical Features

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

## 10. Train/Test Split

We use 80% of the data for training and 20% for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))

## 11. Model 1 — Linear Regression

In [ ]:
linear_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

## 12. Model 2 — Random Forest Regressor

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

## 13. Model Evaluation

**MAE:** lower is better.  
**RMSE:** lower is better.  
**R²:** higher is better; 1.0 is the best possible score.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2 Score': r2}

results = pd.DataFrame([
    evaluate_model('Linear Regression', y_test, y_pred_linear),
    evaluate_model('Random Forest', y_test, y_pred_rf)
])

display(results)

## 14. Select the Best Model

In [ ]:
best_model_name = results.sort_values('R2 Score', ascending=False).iloc[0]['Model']
print('Best model based on R²:', best_model_name)

## 15. Actual vs Predicted Price

In [ ]:
best_predictions = y_pred_rf if best_model_name == 'Random Forest' else y_pred_linear

plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_predictions)
plt.xlabel('Actual Selling Price')
plt.ylabel('Predicted Selling Price')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.show()

## 16. Feature Importance — Random Forest

In [ ]:
rf = rf_model.named_steps['model']
preprocessor_fitted = rf_model.named_steps['preprocessor']
feature_names = preprocessor_fitted.get_feature_names_out()

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

display(feature_importance.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature')
plt.title('Top 10 Feature Importance — Random Forest')
plt.show()

## 17. Predict the Price of a New Car

In [ ]:
new_car = pd.DataFrame({
    'Brand': ['Honda'],
    'Car_Age': [5],
    'Present_Price': [8.5],
    'Kms_Driven': [40000],
    'Fuel_Type': ['Petrol'],
    'Seller_Type': ['Dealer'],
    'Transmission': ['Manual'],
    'Owner': [0]
})

predicted_price = rf_model.predict(new_car)[0]
print(f'Predicted Selling Price: {predicted_price:.2f} lakh')

## 18. Conclusion

The project predicts used-car selling prices using machine learning. The data was cleaned, car age and brand were engineered, categorical features were one-hot encoded, and two regression models were compared using MAE, RMSE, and R². The model with the best evaluation performance is selected as the final model. Random Forest feature importance is used to identify influential input features.